<a href="https://colab.research.google.com/github/Langston-Wang01/Object-Detection-Pointing-System-CV-/blob/main/August_5th_11th_Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch
from torchvision import datasets
import torchvision.transforms as tvt
from torch.utils.data import DataLoader
from torch import nn

Challenge #1 : Convert a regular NumPy array to a Tensor and push it to the Google Colab cloud GPU.

In [ ]:
shape = (5,5)
ones_np = np.ones(shape)
tensor_convert_np = torch.from_numpy(ones_np)
tensor_convert_np.to("cuda")

tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]], device='cuda:0', dtype=torch.float64)

The purpose of assigning the tensor to a GPU is so that we can perform more operations more quickly.

Challenge #2 : Code a simple multilayer model to classify handwritten digits shapes.

In [ ]:
training_data = datasets.MNIST(root = 'Digits', train = True, download = True, transform = tvt.ToTensor())
# extracts data from MNIST under file "Digits" and transforms images into Tensors for a dataloader

MNIST_Loader = DataLoader(training_data, batch_size = 100, shuffle = True)
# creates mini batches for me to train my model


# create a neurla network with two hidden layers
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linearToRelu = nn.Sequential(
        nn.Linear(28*28, 512),
        nn.ReLU(),
        nn.Linear(512, 512),
        nn.ReLU(),
        nn.Linear(512, 10),
    )
    #serves as my hidden layers to final layer.

  def forward(self, x):
    x = x.to('cuda') # so we can perform computations on the weights faster
    layer = self.flatten(x)
    final_activations = self.linearToRelu(layer)
    return final_activations


model = NeuralNetwork()
model.to('cuda')
next(model.parameters()).device

learning_rate = 1e-3

loss_function = nn.CrossEntropyLoss() # uses softmax that normalizes the digits.
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

def training_loop(dataloader, model, loss_function, optimizer):
  model.train() # Good practice to utilize to insure that normalization goes smooth


  for batchNumber, (image, actual_values) in enumerate(dataloader): # This will extact every image from data loader, and compute the activations of the final layer where I later compare it to the actual values.
    prediction = model(image)
    actual_values = actual_values.to('cuda')
    loss = loss_function(prediction, actual_values)

    # Perform back propagation to nudge weights and biases in the correct direciton.
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batchNumber % 100 == 0: # *** used Pytorch example to help print results ***
            loss, current = loss.item(), batchNumber * 100 + len(image)
            print(f"loss: {loss:>7f}  [{current:>5d}/{len(dataloader.dataset):>5d}]")


def testing_loop(dataloader, model, loss_function):
  model.eval() # important for batch normalizaiton
  total_size = len(dataloader.dataset)
  numberOfBatches = len(dataloader)

  test_loss, correct = 0,0

  with torch.no_grad(): # ensures that autograd stop computing gradients whil e testing.
    for image, actual_values in dataloader:
      prediction = model(image)
      actual_values = actual_values.to('cuda')
      test_loss += loss_function(prediction, actual_values).item()
      correct += (prediction.argmax(1) == actual_values).type(torch.float).sum().item()

    test_loss /= numberOfBatches
    correct /= total_size

  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

  # *** used Pytorch example to help print results ***


# With the training and evaluation loops complete, we can now train the model.


for i in range(20): # amount of epoches

   # amount of epoches
   print(f"Epoch {i + 1}\n")
   training_loop(MNIST_Loader, model, loss_function, optimizer)
   testing_loop(MNIST_Loader, model, loss_function)








Epoch 1

loss: 2.299870  [  100/60000]
loss: 2.304796  [10100/60000]
loss: 2.293396  [20100/60000]
loss: 2.292203  [30100/60000]
loss: 2.285357  [40100/60000]
loss: 2.278157  [50100/60000]
Test Error: 
 Accuracy: 17.6%, Avg loss: 2.276028 

Epoch 2

loss: 2.276720  [  100/60000]
loss: 2.271937  [10100/60000]
loss: 2.259303  [20100/60000]
loss: 2.267507  [30100/60000]
loss: 2.261750  [40100/60000]
loss: 2.253338  [50100/60000]
Test Error: 
 Accuracy: 32.3%, Avg loss: 2.244079 

Epoch 3

loss: 2.240747  [  100/60000]
loss: 2.236452  [10100/60000]
loss: 2.234109  [20100/60000]
loss: 2.223064  [30100/60000]
loss: 2.202897  [40100/60000]
loss: 2.193033  [50100/60000]
Test Error: 
 Accuracy: 46.6%, Avg loss: 2.201606 

Epoch 4

loss: 2.194926  [  100/60000]
loss: 2.195529  [10100/60000]
loss: 2.189850  [20100/60000]
loss: 2.177248  [30100/60000]
loss: 2.160902  [40100/60000]
loss: 2.151918  [50100/60000]
Test Error: 
 Accuracy: 54.3%, Avg loss: 2.141952 

Epoch 5

loss: 2.149995  [  100/6000